In [0]:
import pandas as pd;

In [0]:
customers_df = pd.read_csv("/Volumes/mini_project/default/raw_data/customers.csv")
products_df = pd.read_csv("/Volumes/mini_project/default/raw_data/products.csv")
orders_df = pd.read_csv("/Volumes/mini_project/default/raw_data/orders.csv")
order_items_df = pd.read_csv("/Volumes/mini_project/default/raw_data/order_items.csv")

In [0]:
customers_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 800 entries, 0 to 799
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   customer_id    800 non-null    int64 
 1   customer_name  800 non-null    object
 2   email          800 non-null    object
 3   city           800 non-null    object
 4   customer_type  800 non-null    object
dtypes: int64(1), object(4)
memory usage: 31.4+ KB


In [0]:
products_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 700 entries, 0 to 699
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   product_id    700 non-null    int64  
 1   product_name  700 non-null    object 
 2   category      700 non-null    object 
 3   unit_price    700 non-null    float64
dtypes: float64(1), int64(1), object(2)
memory usage: 22.0+ KB


In [0]:
def clean_orders(df):

    cleaned_df = df.copy()
    #issues log
    issues = []
    #missing customer id`s
    missing_customers = cleaned_df["customer_id"].isna().sum()
    cleaned_df["customer_id"] = cleaned_df["customer_id"].fillna(-1)
    issues.append(
        f"Missing customer_id fixed : {missing_customers}"
    )
    #fixing date formats
    cleaned_df["order_date"] = pd.to_datetime(
        cleaned_df["order_date"],
        format="mixed",
        dayfirst=True
    )
    cleaned_df["order_date"] = cleaned_df["order_date"].dt.strftime(
        "%Y-%m-%d %H:%M:%S"
    )
    issues.append("Order dates standardized")

    return cleaned_df, issues

In [0]:
orders_clean_df, order_issues = clean_orders(orders_df)

In [0]:
orders_clean_df.head()

,order_id,customer_id,order_date,status
0,1,711.0,2025-01-26 00:16:36,SHIPPED
1,2,215.0,2025-08-09 15:30:55,PLACED
2,3,782.0,2025-05-31 17:36:43,PLACED
3,4,-1.0,2025-02-14 03:01:30,CANCELLED
4,5,-1.0,2026-02-20 14:05:53,RETURNED


In [0]:
orders_clean_df["customer_id"].isna().sum()

np.int64(0)

In [0]:
orders_clean_df["order_date"].head()

0    2025-01-26 00:16:36
1    2025-08-09 15:30:55
2    2025-05-31 17:36:43
3    2025-02-14 03:01:30
4    2026-02-20 14:05:53
Name: order_date, dtype: object

In [0]:
for issue in order_issues:
    print(issue)

Missing customer_id fixed : 47
Order dates standardized


In [0]:
def clean_products(df):
    cleaned_df = df.copy()
    issues = []
    # Remove leading/trailing spaces
    cleaned_df["product_name"] = cleaned_df["product_name"].str.strip()
    # Convert to Title Case
    cleaned_df["product_name"] = cleaned_df["product_name"].str.title()
    return cleaned_df

In [0]:
products_clean_df = clean_products(products_df)

In [0]:
def validate_emails(df):
    cleaned_df = df.copy()
    # Find invalid emails
    invalid_emails = cleaned_df[
        ~cleaned_df["email"].str.contains(
            r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$",
            regex=True,
            na=False
        )
    ]
    return invalid_emails[["customer_id", "email"]]

In [0]:
invalid_email_df = validate_emails(customers_df)

In [0]:
invalid_email_df

,customer_id,email
106,107,ronaldstephensexample.net
108,109,sbarkerexample.com
219,220,zachary87example.com
330,331,walterskristen
339,340,ncookexample.com
387,388,fheathexample.org
445,446,williamssamuelexample.com
519,520,jhernandezexample.net
649,650,mauriceharrison
683,684,michael77example.com


In [0]:
print("Invalid Emails:", len(invalid_email_df))

Invalid Emails: 10


In [0]:
def check_referential_integrity(orders_df, order_items_df):
    # finding order_ids that exist in order_items but not in orders
    invalid_orders = order_items_df[
        ~order_items_df["order_id"].isin(orders_df["order_id"])
    ]
    return invalid_orders

In [0]:
invalid_orders_df = check_referential_integrity(orders_clean_df,order_items_df)

In [0]:
invalid_orders_df

,order_item_id,order_id,product_id,quantity,discount_percent


In [0]:
print("Invalid Order References :", len(invalid_orders_df))

Invalid Order References : 0


In [0]:
customers_df.loc[~customers_df["email"].str.contains(r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$", na=False), "email"] = "invalid@email.com"

In [0]:
customers_df.to_csv("/Volumes/mini_project/default/cleaned_data/customers_clean.csv", index=False)
products_clean_df.to_csv("/Volumes/mini_project/default/cleaned_data/products_clean.csv", index=False)
orders_clean_df.to_csv("/Volumes/mini_project/default/cleaned_data/orders_clean.csv", index=False)
order_items_df.to_csv("/Volumes/mini_project/default/cleaned_data/order_items_clean.csv", index=False)
print("All cleaned files saved successfully!")

All cleaned files saved successfully!
